# Solving CartPole-v1 with PPO (from scratch)

This notebook demonstrates how to solve the CartPole-v1 environment using a custom implementation of the Proximal Policy Optimization (PPO) algorithm in PyTorch. No external RL libraries are used.

## 1. Install and Import Required Libraries
We will use gymnasium and torch for this implementation.

In [1]:
# Install and import required libraries
import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import gymnasium as gym
except ImportError:
    install("gymnasium")
    import gymnasium as gym

try:
    import torch
except ImportError:
    install("torch")
    import torch

import numpy as np
import torch.nn as nn
import torch.optim as optim

## 2. Set Up CartPole-v1 Environment
We will initialize the CartPole-v1 environment and display its basic information.

In [2]:
# Set up the CartPole-v1 environment
env = gym.make("CartPole-v1")
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Action space: Discrete(2)


## 3. Implement PPO Agent
We will define the policy and value networks, and the PPO update step.

In [3]:
# Policy and Value Networks
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, act_dim)
        )
    def forward(self, x):
        return self.net(x)

class ValueNet(nn.Module):
    def __init__(self, obs_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

## 4. Train PPO Agent
We will train the PPO agent on CartPole-v1.

In [4]:
# Helper functions for PPO
from torch.distributions.categorical import Categorical

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0
    values = values + [0]
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t+1] * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    return advantages

# PPO Training Loop
policy = PolicyNet(obs_dim, act_dim)
value = ValueNet(obs_dim)
policy_optim = optim.Adam(policy.parameters(), lr=3e-4)
value_optim = optim.Adam(value.parameters(), lr=1e-3)

clip_epsilon = 0.2
epochs = 10
steps_per_epoch = 2048
minibatch_size = 64
gamma = 0.99
lam = 0.95

for epoch in range(epochs):
    obs_buf, act_buf, adv_buf, ret_buf, logp_buf = [], [], [], [], []
    val_buf, rew_buf, done_buf = [], [], []
    obs, info = env.reset()
    done = False
    ep_rews = []
    for step in range(steps_per_epoch):
        obs_t = torch.as_tensor(obs, dtype=torch.float32)
        logits = policy(obs_t)
        dist = Categorical(logits=logits)
        action = dist.sample()
        logp = dist.log_prob(action)
        value_pred = value(obs_t).item()
        next_obs, reward, terminated, truncated, info = env.step(action.item())
        done_flag = terminated or truncated
        obs_buf.append(obs)
        act_buf.append(action.item())
        logp_buf.append(logp.item())
        val_buf.append(value_pred)
        rew_buf.append(reward)
        done_buf.append(done_flag)
        obs = next_obs
        ep_rews.append(reward)
        if done_flag:
            obs, info = env.reset()
            ep_rews = []
    adv_buf = compute_gae(rew_buf, val_buf, done_buf, gamma, lam)
    ret_buf = (np.array(adv_buf) + np.array(val_buf)).tolist()
    # Convert buffers to tensors
    obs_tensor = torch.as_tensor(np.array(obs_buf), dtype=torch.float32)
    act_tensor = torch.as_tensor(np.array(act_buf), dtype=torch.int64)
    adv_tensor = torch.as_tensor(np.array(adv_buf), dtype=torch.float32)
    ret_tensor = torch.as_tensor(np.array(ret_buf), dtype=torch.float32)
    logp_old_tensor = torch.as_tensor(np.array(logp_buf), dtype=torch.float32)
    # Normalize advantages
    adv_tensor = (adv_tensor - adv_tensor.mean()) / (adv_tensor.std() + 1e-8)
    # PPO update
    for _ in range(10):
        idx = np.random.permutation(steps_per_epoch)
        for start in range(0, steps_per_epoch, minibatch_size):
            end = start + minibatch_size
            mb_idx = idx[start:end]
            logits = policy(obs_tensor[mb_idx])
            dist = Categorical(logits=logits)
            logp = dist.log_prob(act_tensor[mb_idx])
            ratio = torch.exp(logp - logp_old_tensor[mb_idx])
            surr1 = ratio * adv_tensor[mb_idx]
            surr2 = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * adv_tensor[mb_idx]
            policy_loss = -torch.min(surr1, surr2).mean()
            value_pred = value(obs_tensor[mb_idx]).squeeze()
            value_loss = ((ret_tensor[mb_idx] - value_pred) ** 2).mean()
            policy_optim.zero_grad()
            policy_loss.backward()
            policy_optim.step()
            value_optim.zero_grad()
            value_loss.backward()
            value_optim.step()
    print(f"Epoch {epoch+1}/{epochs} complete.")

Epoch 1/10 complete.
Epoch 2/10 complete.
Epoch 2/10 complete.
Epoch 3/10 complete.
Epoch 3/10 complete.
Epoch 4/10 complete.
Epoch 4/10 complete.
Epoch 5/10 complete.
Epoch 5/10 complete.
Epoch 6/10 complete.
Epoch 6/10 complete.
Epoch 7/10 complete.
Epoch 7/10 complete.
Epoch 8/10 complete.
Epoch 8/10 complete.
Epoch 9/10 complete.
Epoch 9/10 complete.
Epoch 10/10 complete.
Epoch 10/10 complete.


In [5]:
# Optionally, visualize one episode
obs, info = env.reset()
done = False
while not done:
    obs_t = torch.as_tensor(obs, dtype=torch.float32)
    logits = policy(obs_t)
    dist = Categorical(logits=logits)
    action = dist.probs.argmax().item()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    env.render()
env.close()

/home/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.10/site-packages/gymnasium/envs/classic_control/cartpole.py:250: UserWarning: WARN: You are calling render method without specifying any render mode. You can specify the render_mode at initialization, e.g. gym.make("CartPole-v1", render_mode="rgb_array")
  gym.logger.warn(


In [6]:
# Evaluate the trained agent
num_episodes = 20
rewards = []
for ep in range(num_episodes):
    obs, info = env.reset()
    done = False
    ep_reward = 0
    while not done:
        obs_t = torch.as_tensor(obs, dtype=torch.float32)
        logits = policy(obs_t)
        dist = Categorical(logits=logits)
        action = dist.probs.argmax().item()
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        ep_reward += reward
    rewards.append(ep_reward)
    print(f"Episode {ep+1}: Reward = {ep_reward}")
print(f"Mean reward over {num_episodes} episodes: {np.mean(rewards):.2f}")
if np.mean(rewards) >= 475:
    print("CartPole-v1 solved!")
else:
    print("CartPole-v1 not solved. Try training for more epochs.")

Episode 1: Reward = 290.0
Episode 2: Reward = 269.0
Episode 3: Reward = 362.0
Episode 4: Reward = 493.0
Episode 5: Reward = 293.0
Episode 6: Reward = 364.0
Episode 4: Reward = 493.0
Episode 5: Reward = 293.0
Episode 6: Reward = 364.0
Episode 7: Reward = 500.0
Episode 8: Reward = 265.0
Episode 9: Reward = 500.0
Episode 7: Reward = 500.0
Episode 8: Reward = 265.0
Episode 9: Reward = 500.0
Episode 10: Reward = 263.0
Episode 11: Reward = 329.0
Episode 12: Reward = 286.0
Episode 10: Reward = 263.0
Episode 11: Reward = 329.0
Episode 12: Reward = 286.0
Episode 13: Reward = 500.0
Episode 14: Reward = 298.0
Episode 15: Reward = 323.0
Episode 16: Reward = 277.0
Episode 13: Reward = 500.0
Episode 14: Reward = 298.0
Episode 15: Reward = 323.0
Episode 16: Reward = 277.0
Episode 17: Reward = 490.0
Episode 18: Reward = 500.0
Episode 17: Reward = 490.0
Episode 18: Reward = 500.0
Episode 19: Reward = 386.0
Episode 20: Reward = 500.0
Mean reward over 20 episodes: 374.40
CartPole-v1 not solved. Try train